[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyeongHyeonChoi/AI-Stock-Price-Prediction
/blob/main/AI%20Stock%20Price%20Prediction(2023004384%20%EB%AC%BC%EB%A6%AC%ED%95%99%EA%B3%BC%20%EC%B5%9C%EB%AA%85%ED%98%84).ipynb)

In [ ]:
from google.colab import files
# csv data from https://data.krx.co.kr/contents/MDC/MAIN/main/index.cmd #
uploaded = files.upload()

In [ ]:
import os
print(os.listdir())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam


class StockPredictor:
    def __init__(self, file_name, window_size=5):
        self.file_name = file_name
        self.window_size = window_size
        self.data = None

        # Scaler for input features
        self.scaler = MinMaxScaler()

        # Scaler for target closing price
        self.target_scaler = MinMaxScaler()

        self.model = None

    def load_data(self):
        """
        Load and clean KRX Korean CSV file.
        """
        self.data = pd.read_csv(self.file_name, encoding="cp949")

        # Remove spaces from column names
        self.data.columns = self.data.columns.str.strip()

        # Rename Korean column names to English column names
        self.data = self.data.rename(columns={
            "일자": "Date",
            "날짜": "Date",
            "시가": "Open",
            "고가": "High",
            "저가": "Low",
            "종가": "Close",
            "거래량": "Volume"
        })

        # Use only necessary columns
        required_columns = ["Date", "Open", "High", "Low", "Close", "Volume"]
        self.data = self.data[required_columns]

        # Convert date format
        self.data["Date"] = pd.to_datetime(self.data["Date"])

        # Convert numeric columns
        numeric_columns = ["Open", "High", "Low", "Close", "Volume"]

        for col in numeric_columns:
            self.data[col] = (
                self.data[col]
                .astype(str)
                .str.replace(",", "", regex=False)
                .astype(float)
            )

        # Remove duplicate dates
        self.data = self.data.drop_duplicates(subset=["Date"])

        # Sort by date from past to present
        self.data = self.data.sort_values("Date")

        # Remove missing values
        self.data = self.data.dropna()

        print("Data loading completed.")
        print(self.data.head())
        print()
        print("Number of data points:", len(self.data))

    def make_features(self):
        """
        Add moving averages and price change features.
        """
        self.data["MA5"] = self.data["Close"].rolling(window=5).mean()
        self.data["MA10"] = self.data["Close"].rolling(window=10).mean()
        self.data["Change"] = self.data["Close"].pct_change()

        self.data = self.data.dropna()

        print()
        print("Feature engineering completed.")
        print(self.data[["Date", "Close", "MA5", "MA10", "Change"]].head())

    def create_dataset(self):
        """
        Create input X using recent window_size days,
        and target y using the next day's closing price.
        """
        feature_columns = [
            "Open", "High", "Low", "Close", "Volume",
            "MA5", "MA10", "Change"
        ]

        feature_data = self.data[feature_columns].values
        close_data = self.data[["Close"]].values

        # Normalize input features
        scaled_features = self.scaler.fit_transform(feature_data)

        # Normalize target closing price
        scaled_close = self.target_scaler.fit_transform(close_data)

        X = []
        y = []

        for i in range(self.window_size, len(scaled_features)):
            X.append(scaled_features[i - self.window_size:i].flatten())
            y.append(scaled_close[i][0])

        X = np.array(X)
        y = np.array(y)

        print()
        print("Training dataset creation completed.")
        print("X shape:", X.shape)
        print("y shape:", y.shape)

        return X, y

    def build_model(self, input_size):
        """
        Build artificial neural network model.
        """
        self.model = Sequential()

        self.model.add(Dense(64, activation="relu", input_shape=(input_size,)))
        self.model.add(Dense(32, activation="relu"))
        self.model.add(Dense(16, activation="relu"))
        self.model.add(Dense(1))

        self.model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss="mse"
        )

        print()
        print("Model construction completed.")
        self.model.summary()

    def train_model(self, X_train, y_train, X_test, y_test):
        """
        Train the model.
        """
        print()
        print("Model training started.")

        history = self.model.fit(
            X_train,
            y_train,
            epochs=100,
            batch_size=16,
            validation_data=(X_test, y_test),
            verbose=1
        )

        return history

    def evaluate_model(self, X_test, y_test):
        """
        Compare actual and predicted closing prices and calculate errors.
        """
        scaled_predictions = self.model.predict(X_test).flatten()

        predictions = self.target_scaler.inverse_transform(
            scaled_predictions.reshape(-1, 1)
        ).flatten()

        y_test_real = self.target_scaler.inverse_transform(
            y_test.reshape(-1, 1)
        ).flatten()

        mae = mean_absolute_error(y_test_real, predictions)
        mse = mean_squared_error(y_test_real, predictions)
        rmse = np.sqrt(mse)

        print()
        print("Model Evaluation Results")
        print(f"MAE  : {mae:.2f} KRW")
        print(f"MSE  : {mse:.2f}")
        print(f"RMSE : {rmse:.2f} KRW")

        return predictions, mae, mse, rmse, y_test_real

    def plot_loss(self, history):
        """
        Plot training loss and validation loss.
        """
        plt.figure(figsize=(10, 5))
        plt.plot(history.history["loss"], label="Training Loss")
        plt.plot(history.history["val_loss"], label="Validation Loss")
        plt.title("Training Loss and Validation Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid()
        plt.show()

    def plot_result(self, y_test_real, predictions):
        """
        Plot actual and predicted closing prices.
        """
        plt.figure(figsize=(12, 5))
        plt.plot(y_test_real, label="Real Close Price")
        plt.plot(predictions, label="Predicted Close Price")
        plt.title("Real Stock Price vs Predicted Stock Price")
        plt.xlabel("Test Data Index")
        plt.ylabel("Close Price")
        plt.legend()
        plt.grid()
        plt.show()

    def predict_next_day(self):
        """
        Predict the closing price for the next trading day.
        """
        feature_columns = [
            "Open", "High", "Low", "Close", "Volume",
            "MA5", "MA10", "Change"
        ]

        recent_data = self.data[feature_columns].values[-self.window_size:]
        scaled_recent_data = self.scaler.transform(recent_data)

        X_recent = scaled_recent_data.flatten().reshape(1, -1)

        scaled_predicted_price = self.model.predict(X_recent, verbose=0)[0][0]

        predicted_price = self.target_scaler.inverse_transform(
            np.array([[scaled_predicted_price]])
        )[0][0]

        print()
        print(f"Predicted closing price for the next trading day: {predicted_price:.2f} KRW")

        return predicted_price

    def predict_future_days(self, days=20):

        feature_columns = [
            "Open", "High", "Low", "Close", "Volume",
            "MA5", "MA10", "Change"
        ]

        future_data = self.data.copy()
        future_prices = []
        future_dates = []

        if len(future_data) < self.window_size:
            raise ValueError("Not enough data for future prediction.")

        for _ in range(days):
            recent_data = future_data[feature_columns].values[-self.window_size:]
            scaled_recent_data = self.scaler.transform(recent_data)

            X_recent = scaled_recent_data.flatten().reshape(1, -1)

            scaled_predicted_close = self.model.predict(X_recent, verbose=0)[0][0]

            predicted_close = self.target_scaler.inverse_transform(
                np.array([[scaled_predicted_close]])
            )[0][0]

            last_date = future_data["Date"].iloc[-1]
            next_date = last_date + pd.Timedelta(days=1)

            # Skip weekends
            while next_date.weekday() >= 5:
                next_date += pd.Timedelta(days=1)

            future_prices.append(predicted_close)
            future_dates.append(next_date)

            new_row = {
                "Date": next_date,
                "Open": predicted_close,
                "High": predicted_close,
                "Low": predicted_close,
                "Close": predicted_close,
                "Volume": future_data["Volume"].iloc[-1],
                "MA5": np.nan,
                "MA10": np.nan,
                "Change": np.nan
            }

            future_data = pd.concat(
                [future_data, pd.DataFrame([new_row])],
                ignore_index=True
            )

            future_data["MA5"] = future_data["Close"].rolling(window=5).mean()
            future_data["MA10"] = future_data["Close"].rolling(window=10).mean()
            future_data["Change"] = future_data["Close"].pct_change()

            # Do not drop rows during iterative future prediction
            future_data[["MA5", "MA10", "Change"]] = (
                future_data[["MA5", "MA10", "Change"]]
                .bfill()
                .ffill()
            )

        return future_dates, future_prices

    def plot_future_prediction(self, days=20):
        """
        Plot future predicted closing prices after real stock price data.
        """
        future_dates, future_prices = self.predict_future_days(days=days)

        plt.figure(figsize=(12, 5))

        plt.plot(
            self.data["Date"],
            self.data["Close"],
            label="Real Close Price"
        )

        plt.plot(
            future_dates,
            future_prices,
            linestyle="--",
            marker="o",
            label="Future Predicted Price"
        )

        plt.title("Real Stock Price and Future Predicted Stock Price")
        plt.xlabel("Date")
        plt.ylabel("Close Price")
        plt.legend()
        plt.grid()
        plt.xticks(rotation=45)
        plt.show()

        print()
        print(f"Prediction of closing prices for the next {days} trading days")
        for date, price in zip(future_dates, future_prices):
            print(f"{date.date()} predicted closing price: {price:.2f} KRW")

        return future_dates, future_prices

    def run(self):
        """
        Full execution pipeline.
        """
        self.load_data()
        self.make_features()

        X, y = self.create_dataset()

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            shuffle=False
        )

        print()
        print("Train/Test split completed.")
        print("Train data:", X_train.shape)
        print("Test data :", X_test.shape)

        self.build_model(input_size=X_train.shape[1])

        history = self.train_model(X_train, y_train, X_test, y_test)

        predictions, mae, mse, rmse, y_test_real = self.evaluate_model(X_test, y_test)

        self.plot_loss(history)
        self.plot_result(y_test_real, predictions)

        next_price = self.predict_next_day()

        future_dates, future_prices = self.plot_future_prediction(days=20)

        return predictions, mae, mse, rmse, next_price, future_dates, future_prices

In [ ]:
samsung_predictor = StockPredictor("Sam.csv", window_size=5)
samsung_result = samsung_predictor.run()

In [ ]:
samsung_predictor = StockPredictor("SK.csv", window_size=5)
samsung_result = samsung_predictor.run()